# Qwen3.5-2B — dostrajanie modelu metodą LoRA i ewaluacja

W tym notebooku przedstawiono eksperyment polegający na dostrojeniu modelu językowego Qwen3.5-2B do zadań związanych z informacją medyczną.

Do treningu wykorzystano zbiór danych `medalpaca/medical_meadow_wikidoc_patient_information` oraz metodę LoRA zaimplementowaną w narzędziu MS Swift.

Notebook obejmuje przygotowanie danych, sprawdzenie modelu przed treningiem, dostrajanie, ponowną ewaluację oraz porównanie wyników.

Do oceny wykorzystano metryki ROUGE-1, ROUGE-2, ROUGE-L i BLEU, a także średnią liczbę słów w generowanych odpowiedziach.

Eksperyment przeprowadzono w środowisku Kaggle z wykorzystaniem karty graficznej Tesla T4.

## 1. Instalacja wymaganych bibliotek

Na początku instalowane są biblioteki potrzebne do przygotowania danych, dostrajania modelu i oceny jego odpowiedzi.

Wykorzystywane narzędzia to między innymi MS Swift, Datasets, ROUGE Score i SacreBLEU.

In [9]:
!pip uninstall -y torchao
!pip install -U ms-swift datasets rouge_score sacrebleu "qwen_vl_utils>=0.0.14" "transformers>=5.2.0" peft

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 143.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 135.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━

## 2. Sprawdzenie środowiska

Przed rozpoczęciem eksperymentu sprawdzane są wersje najważniejszych bibliotek, dostępność CUDA oraz parametry karty graficznej.

Trening modelu wymaga dostępu do GPU. W tym eksperymencie wykorzystano kartę NVIDIA Tesla T4.

In [10]:
import os
import re
import json
import glob
import shutil
import numpy as np
import torch
import transformers
import swift
import datasets
import rouge_score
import sacrebleu

from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("MS Swift:", swift.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    raise RuntimeError("No GPU detected. In Kaggle, open Settings → Accelerator → GPU.")

PyTorch: 2.11.0+cu128
Transformers: 5.19.0
MS Swift: 4.6.0
Datasets: 4.8.4
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


## 3. Wczytanie i przegląd zbioru danych

Dane pochodzą ze zbioru `medalpaca/medical_meadow_wikidoc_patient_information`, udostępnionego na platformie Hugging Face.

W tej części zbiór jest wczytywany, a następnie wyświetlana jest jego struktura i przykładowy rekord. Pozwala to sprawdzić, jakie informacje zawierają dane i w jakim formacie są zapisane.

In [3]:
from datasets import load_dataset

dataset = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information")
print(dataset)

example = dataset["train"][0]
for key, value in example.items():
    print(f"\\n{key.upper()}:\\n{value}")

README.md:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

medical_meadow_wikidoc_patient_info.json: reconstructing file:   0%|          |  0.00B / 3.49MB            

medical_meadow_wikidoc_patient_info.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 5942
    })
})
\nINPUT:\nWhat are the symptoms of Allergy?
\nOUTPUT:\nAllergy symptoms vary, but may include:
Breathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing
What part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:
Allergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may cause itchy, watery, red, swollen eyes. Eating something you are allergic to can cause nausea, vomiting, abdominal pain, cramping, diarrhea, or a severe, life-threatening reaction. Allergens that touch the skin can cause a skin rash, hives, itching, blister

## 4. Filtrowanie danych i podział na zbiór treningowy oraz testowy

Przed treningiem dane zostały przefiltrowane, aby usunąć niekompletne lub zbyt długie przykłady.

Zastosowano następujące warunki:
- pole `input` nie może być puste;
- pole `output` musi zawierać co najmniej 20 znaków;
- łączna długość `input` i `output` nie może przekraczać 1500 znaków.

Po filtrowaniu dane podzielono na zbiór treningowy (90%) i testowy (10%). Użycie `seed=42` pozwala zachować powtarzalność podziału.

In [4]:
def is_valid(example):
    input_text = example["input"] or ""
    output_text = example["output"] or ""
    return (
        len(input_text.strip()) > 0
        and len(output_text.strip()) >= 20
        and len(input_text) + len(output_text) <= 1500
    )

filtered_dataset = dataset["train"].filter(is_valid)

print("Original examples:", len(dataset["train"]))
print("After filtering:", len(filtered_dataset))
print("Removed:", len(dataset["train"]) - len(filtered_dataset))

split_dataset = filtered_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Filter:   0%|          | 0/5942 [00:00<?, ? examples/s]

Original examples: 5942
After filtering: 5720
Removed: 222
Training examples: 5148
Test examples: 572


## 5. Przygotowanie danych w formacie Messages i zapis do plików JSONL

Dane zostały przekształcone do formatu konwersacyjnego, w którym każda próbka zawiera trzy role:
- `system` — instrukcja określająca sposób odpowiadania;
- `user` — pytanie;
- `assistant` — odpowiedź referencyjna.

Tak przygotowane dane zapisano w formacie JSONL, który jest wykorzystywany w dalszym procesie treningu i ewaluacji.

In [6]:
def convert_to_messages(example):
    return {
        "messages": [
            {"role": "system", "content": example["instruction"]},
            {"role": "user", "content": example["input"]},
            {"role": "assistant", "content": example["output"]}
        ]
    }

train_messages = train_dataset.map(convert_to_messages)
test_messages = test_dataset.map(convert_to_messages)

os.makedirs("/kaggle/working/data", exist_ok=True)

def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for example in data:
            f.write(
                json.dumps(
                    {"messages": example["messages"]},
                    ensure_ascii=False
                ) + "\n"
            )

save_jsonl(train_messages, "/kaggle/working/data/train.jsonl")
save_jsonl(test_messages, "/kaggle/working/data/test.jsonl")

print("First JSONL record:")
print(json.dumps(train_messages[0], ensure_ascii=False, indent=2))
print("\nTraining examples:", len(train_messages))
print("Test examples:", len(test_messages))

First JSONL record:
{
  "input": "What are the symptoms of Merkel cell cancer?",
  "output": "Rapid growing, painless and small mass that typically located on a sun exposed area of the body",
  "instruction": "Answer this question truthfully",
  "messages": [
    {
      "content": "Answer this question truthfully",
      "role": "system"
    },
    {
      "content": "What are the symptoms of Merkel cell cancer?",
      "role": "user"
    },
    {
      "content": "Rapid growing, painless and small mass that typically located on a sun exposed area of the body",
      "role": "assistant"
    }
  ]
}

Training examples: 5148
Test examples: 572


## 6. Funkcje do generowania odpowiedzi i ewaluacji

W tej sekcji zdefiniowano funkcje wykorzystywane do oceny modelu bazowego i modelu po dostrojeniu.

Funkcja `build_requests` przygotowuje zapytania w formacie wymaganym przez MS Swift, a `evaluate_engine` generuje odpowiedzi modelu dla danych testowych.

Funkcja `clean_answer` usuwa wewnętrzne znaczniki rozumowania, natomiast `calculate_metrics` oblicza:
- ROUGE-1;
- ROUGE-2;
- ROUGE-L;
- BLEU;
- średnią liczbę słów w odpowiedziach.

In [11]:
import re
import numpy as np
from tqdm.auto import tqdm
from datasets import Dataset
from swift.infer_engine import InferRequest, RequestConfig

def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()

def build_requests(batch):
    requests = []
    for example in batch:
        messages = example["messages"]
        requests.append(
            InferRequest(messages=[
                {"role": "system", "content": messages[0]["content"]},
                {"role": "user", "content": messages[1]["content"]}
            ])
        )
    return requests

def evaluate_engine(engine, eval_data, label):
    predictions, references = [], []
    config = RequestConfig(max_tokens=200, temperature=0, seed=42)
    batch_size = 8

    for start in tqdm(range(0, len(eval_data), batch_size), desc=label):
        end = min(start + batch_size, len(eval_data))
        batch = eval_data.select(range(start, end))
        responses = engine.infer(build_requests(batch), request_config=config)

        for example, response in zip(batch, responses):
            answer = response.choices[0].message.content or ""
            predictions.append(clean_answer(answer))
            references.append(example["messages"][2]["content"])

    return predictions, references

def calculate_metrics(predictions, references):
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"], use_stemmer=True
    )
    r1, r2, rl = [], [], []

    for prediction, reference in zip(predictions, references):
        scores = scorer.score(reference, prediction)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    return {
        "ROUGE-1": float(np.mean(r1)),
        "ROUGE-2": float(np.mean(r2)),
        "ROUGE-L": float(np.mean(rl)),
        "BLEU": float(sacrebleu.corpus_bleu(predictions, [references]).score),
        "Average words": float(np.mean([len(x.split()) for x in predictions]))
    }

eval_dataset = Dataset.from_list(test_data)

print("Evaluation examples:", len(eval_dataset))
print("First evaluation example:", eval_dataset[0]["messages"][1]["content"])

Evaluation examples: 572
First evaluation example: What to expect if I have Tuberculosis  (Outlook/Prognosis)?


## 7. Ewaluacja modelu bazowego

Przed dostrojeniem oceniono model bazowy Qwen3.5-2B na przygotowanym zbiorze testowym.

Wyniki ewaluacji oraz wygenerowane odpowiedzi zostały zapisane, aby można je było wykorzystać później do porównania z modelem po dostrojeniu.

In [12]:
baseline_metrics = baseline_data["metrics"]
baseline_predictions = baseline_data["predictions"]
baseline_references = baseline_data["references"]

print("BASELINE METRICS")
for metric, value in baseline_metrics.items():
    print(f"{metric}: {value:.4f}")

print("Baseline examples:", len(baseline_predictions))

BASELINE METRICS
ROUGE-1: 0.2477
ROUGE-2: 0.0553
ROUGE-L: 0.1389
BLEU: 1.7145
Average words: 134.5070
Baseline examples: 572


In [14]:
import zipfile
import os
import shutil

source = "/kaggle/input/datasets/andi11andi/medical-meadow-recovery/recovery_files/baseline_results.json"
working_copy = "/kaggle/working/baseline_results.json"
backup = "/kaggle/working/baseline_backup.zip"

assert os.path.exists(source), "Baseline JSON not found in Input!"

shutil.copy2(source, working_copy)

with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(working_copy, arcname="baseline_results.json")

print("Backup size:", round(os.path.getsize(backup) / 1024, 2), "KB")
print("Backup created successfully!")

Backup size: 238.26 KB
Backup created successfully!


In [15]:
import os

for filename in [
    "baseline_results.json",
    "baseline_backup.zip",
    "data/train.jsonl",
    "data/test.jsonl"
]:
    path = "/kaggle/working/" + filename
    print(
        filename,
        os.path.exists(path),
        os.path.getsize(path) if os.path.exists(path) else "NOT FOUND"
    )

baseline_results.json True 744393
baseline_backup.zip True 243979
data/train.jsonl False NOT FOUND
data/test.jsonl False NOT FOUND


In [16]:
import os
import shutil

source_dir = "/kaggle/input/datasets/andi11andi/medical-meadow-recovery/recovery_files"
target_dir = "/kaggle/working/data"

os.makedirs(target_dir, exist_ok=True)

shutil.copy2(
    os.path.join(source_dir, "train.jsonl"),
    os.path.join(target_dir, "train.jsonl")
)

shutil.copy2(
    os.path.join(source_dir, "test.jsonl"),
    os.path.join(target_dir, "test.jsonl")
)

for filename in ["train.jsonl", "test.jsonl"]:
    path = os.path.join(target_dir, filename)
    print(filename, os.path.exists(path), os.path.getsize(path))

train.jsonl True 3052230
test.jsonl True 339214


## 8. Zwolnienie pamięci GPU przed treningiem

Przed rozpoczęciem dostrajania zwalniana jest pamięć GPU zajmowana wcześniej przez model bazowy.

Model zostanie ponownie wczytany przez MS Swift podczas procesu treningowego. Dzięki temu dostępna pamięć GPU może zostać wykorzystana do dostrajania modelu.

In [17]:
import gc
import torch

for name in ["base_engine", "model", "processor", "template"]:
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cache cleared.")

GPU memory cache cleared.


## 9. Dostrajanie modelu metodą LoRA

Model Qwen3.5-2B został dostrojony przy użyciu metody LoRA za pomocą narzędzia MS Swift.

W treningu wykorzystano przygotowany zbiór danych treningowych. Zastosowano jedną epokę treningową, rangę LoRA równą 8 oraz współczynnik uczenia 0.0001.

Po zakończeniu treningu zapisano checkpoint modelu, a następnie utworzono archiwum zawierające adapter LoRA. Sprawdzono również poprawność archiwum i możliwość jego odczytu.

In [18]:
!swift sft \
    --model Qwen/Qwen3.5-2B \
    --tuner_type lora \
    --dataset /kaggle/working/data/train.jsonl \
    --torch_dtype float16 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 16 \
    --target_modules all-linear \
    --num_train_epochs 1 \
    --max_length 512 \
    --gradient_checkpointing true \
    --save_steps 200 \
    --save_total_limit 2 \
    --logging_steps 10 \
    --output_dir /kaggle/working/qwen_lora \
    --use_hf true

run sh: `/usr/bin/python3 /usr/local/lib/python3.13/dist-packages/swift/cli/sft.py --model Qwen/Qwen3.5-2B --tuner_type lora --dataset /kaggle/working/data/train.jsonl --torch_dtype float16 --per_device_train_batch_size 1 --gradient_accumulation_steps 8 --learning_rate 1e-4 --lora_rank 8 --lora_alpha 16 --target_modules all-linear --num_train_epochs 1 --max_length 512 --gradient_checkpointing true --save_steps 200 --save_total_limit 2 --logging_steps 10 --output_dir /kaggle/working/qwen_lora --use_hf true`
[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 12 files:   0%|                                 | 0/12 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to

In [19]:
import os
import zipfile

checkpoint_dir = "/kaggle/working/qwen_lora/v0-20261102-080415/checkpoint-644"
adapter_zip = "/kaggle/working/qwen_lora_adapter.zip"

assert os.path.exists(checkpoint_dir), "Checkpoint directory not found!"

with zipfile.ZipFile(adapter_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(checkpoint_dir):
        for filename in files:
            file_path = os.path.join(root, filename)
            arcname = os.path.relpath(file_path, checkpoint_dir)
            z.write(file_path, arcname)

with zipfile.ZipFile(adapter_zip, "r") as z:
    result = z.testzip()

print("Adapter archive size:", round(os.path.getsize(adapter_zip) / (1024 ** 2), 2), "MB")
print("Archive integrity:", "OK" if result is None else result)
print("Files in archive:", len(zipfile.ZipFile(adapter_zip).namelist()))

Adapter archive size: 88.69 MB
Archive integrity: OK
Files in archive: 11


## 10. Wczytanie modelu bazowego i adaptera LoRA

Po zakończeniu treningu ponownie wczytywany jest model Qwen3.5-2B oraz zapisany adapter LoRA.

Adapter zostaje dołączony do modelu bazowego przy użyciu biblioteki PEFT. Tak przygotowany model jest wykorzystywany w dalszej ewaluacji na tym samym zbiorze testowym, który wykorzystano do oceny modelu bazowego.

In [23]:
import torch
from swift.model import get_model_processor
from swift.template import get_template
from swift.infer_engine import TransformersEngine
from peft import PeftModel

model_id = "Qwen/Qwen3.5-2B"
adapter_path = "/kaggle/working/qwen_lora/v0-20261102-080415/checkpoint-644"

model, processor = get_model_processor(
    model_id,
    torch_dtype=torch.float16,
    use_hf=True
)

model = PeftModel.from_pretrained(model, adapter_path)

template = get_template(
    processor,
    enable_thinking=False
)

ft_engine = TransformersEngine(
    model,
    template=template
)

print("Base model and LoRA adapter loaded successfully.")

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/mode

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
[INFO:swift] Setting ROOT_IMAGE_DIR: None. You can adjust this hyperparameter through the environment variable: `ROOT_IMAGE_DIR`.
[INFO:swift] Setting QWENVL_BBOX_FORMAT: legacy. You can adjust this hyperparameter through the environment variable: `QWENVL_BBOX_FORMAT`.


Base model and LoRA adapter loaded successfully.


## 11. Ewaluacja modelu po dostrojeniu

Po zakończeniu treningu model bazowy został połączony z adapterem LoRA. Następnie przeprowadzono generowanie odpowiedzi i ocenę modelu na tym samym zbiorze testowym, który wykorzystano do ewaluacji modelu bazowego.

Dla każdego przykładu zapisano wygenerowaną odpowiedź oraz odpowiedź referencyjną. Wyniki wykorzystano do obliczenia metryk ROUGE, BLEU i średniej liczby słów w odpowiedziach.

In [31]:
import json
from tqdm.auto import tqdm
from swift.infer_engine import RequestConfig

results_path = "/kaggle/working/finetuned_predictions_partial.json"

predictions = []
references = []

config = RequestConfig(
    max_tokens=200,
    temperature=0,
    seed=42
)

batch_size = 8

for start in tqdm(range(0, len(eval_dataset), batch_size), desc="Fine-tuned evaluation"):
    end = min(start + batch_size, len(eval_dataset))
    batch = eval_dataset.select(range(start, end))

    responses = ft_engine.infer(
        build_requests(batch),
        request_config=config
    )

    for example, response in zip(batch, responses):
        prediction = response.choices[0].message.content or ""
        predictions.append(clean_answer(prediction))
        references.append(example["messages"][2]["content"])

    with open(results_path, "w", encoding="utf-8") as f:
        json.dump({
            "predictions": predictions,
            "references": references
        }, f, ensure_ascii=False, indent=2)

print("\nEvaluation generation completed.")
print("Total predictions:", len(predictions))
print("Total references:", len(references))
print("Saved to:", results_path)

Fine-tuned evaluation:   0%|          | 0/72 [00:00<?, ?it/s]



  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:03<00:21,  3.11s/it]

 25%|██▌       | 2/8 [00:22<01:17, 12.92s/it]

 38%|███▊      | 3/8 [00:24<00:39,  7.97s/it]

 50%|█████     | 4/8 [00:43<00:49, 12.30s/it]

 62%|██████▎   | 5/8 [00:46<00:26,  8.97s/it]

 75%|███████▌  | 6/8 [00:48<00:12,  6.35s/it]

 88%|████████▊ | 7/8 [00:52<00:05,  5.55s/it]

100%|██████████| 8/8 [01:07<00:00,  8.40s/it]


  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:03<00:21,  3.05s/it]

 25%|██▌       | 2/8 [00:22<01:14, 12.46s/it]

 38%|███▊      | 3/8 [00:29<00:50, 10.16s/it]

 50%|█████     | 4/8 [00:48<00:54, 13.73s/it]

 62%|██████▎   | 5/8 [00:52<00:30, 10.03s/it]

 75%|███████▌  | 6/8 [01:07<00:23, 11.67s/it]

 88%|████████▊ | 7/8 [01:08<00:08,  8.46s/it]

100%|██████████| 8/8 [01:10<00:00,  8.78s/it]


  0%|          | 0/8 [00:00<?, ?it/s]

 12%|█▎        | 1/8 [00:17<02:01, 17.41s/it]

 25%|██▌       | 2/8 [00:23<01:05, 10.98s/it]

 38%|███▊      | 3/8 [00:28<00:39


Evaluation generation completed.
Total predictions: 572
Total references: 572
Saved to: /kaggle/working/finetuned_predictions_partial.json


In [32]:
import json
import sacrebleu

results_path = "/kaggle/working/finetuned_predictions_partial.json"

with open(results_path, "r", encoding="utf-8") as f:
    finetuned_data = json.load(f)

finetuned_predictions = finetuned_data["predictions"]
finetuned_references = finetuned_data["references"]

print("Predictions:", len(finetuned_predictions))
print("References:", len(finetuned_references))

finetuned_metrics = calculate_metrics(
    finetuned_predictions,
    finetuned_references
)

print("\nFINE-TUNED MODEL METRICS")
for metric, value in finetuned_metrics.items():
    print(f"{metric}: {value:.4f}")

final_results_path = "/kaggle/working/finetuned_results.json"

with open(final_results_path, "w", encoding="utf-8") as f:
    json.dump({
        "predictions": finetuned_predictions,
        "references": finetuned_references,
        "metrics": finetuned_metrics
    }, f, ensure_ascii=False, indent=2)

print("\nResults saved to:", final_results_path)

Predictions: 572
References: 572

FINE-TUNED MODEL METRICS
ROUGE-1: 0.2705
ROUGE-2: 0.1068
ROUGE-L: 0.2048
BLEU: 5.1660
Average words: 58.3409

Results saved to: /kaggle/working/finetuned_results.json


## 12. Porównanie wyników ewaluacji

W tej części zestawiono wyniki modelu bazowego i modelu po dostrojeniu.

Porównanie obejmuje metryki ROUGE-1, ROUGE-2, ROUGE-L i BLEU, a także średnią liczbę słów w wygenerowanych odpowiedziach.

Tabela pozwala ocenić, jak dostrojenie wpłynęło na podobieństwo odpowiedzi modelu do odpowiedzi referencyjnych oraz na długość generowanych tekstów.

In [33]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": list(baseline_metrics.keys()),
    "Base model": list(baseline_metrics.values()),
    "Fine-tuned model": [finetuned_metrics[k] for k in baseline_metrics.keys()]
})
comparison["Change"] = comparison["Fine-tuned model"] - comparison["Base model"]
display(comparison.round(4))

,Metric,Base model,Fine-tuned model,Change
0,ROUGE-1,0.2477,0.2705,0.0228
1,ROUGE-2,0.0553,0.1068,0.0515
2,ROUGE-L,0.1389,0.2048,0.0658
3,BLEU,1.7145,5.1660,3.4515
4,Average words,134.5070,58.3409,-76.1661


## 13. Porównanie przykładowych odpowiedzi przed i po dostrojeniu

Poniżej przedstawiono trzy przykłady odpowiedzi wygenerowanych przez model bazowy i model po dostrojeniu.

Każdy przykład zawiera pytanie, odpowiedź referencyjną oraz odpowiedzi obu modeli. Pozwala to porównać nie tylko wyniki metryk, ale również sposób generowania odpowiedzi i ich zawartość.

In [35]:
import json

with open("/kaggle/working/baseline_results.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)

with open("/kaggle/working/finetuned_results.json", "r", encoding="utf-8") as f:
    finetuned_data = json.load(f)

for i in [0, 10, 50]:
    print("\n" + "=" * 80)
    print("EXAMPLE", i + 1)
    print("\nQUESTION:")
    print(test_data[i]["messages"][1]["content"])

    print("\nREFERENCE:")
    print(baseline_data["references"][i])

    print("\nBASELINE:")
    print(baseline_data["predictions"][i])

    print("\nFINE-TUNED:")
    print(finetuned_data["predictions"][i])


EXAMPLE 1

QUESTION:
What to expect if I have Tuberculosis  (Outlook/Prognosis)?

REFERENCE:
Symptoms often improve in 2 - 3 weeks. A chest x-ray will not show this improvement until later. The outlook is excellent if pulmonary TB is diagnosed early and treatment is begun quickly.
The prognosis of multidrug-resistant TB is worse than that of tuberculosis susceptible to the regular common treatment. So it is necessary to do a drug susceptibility test and to monitor adherence to the treatment regimen to avoid new drug resistances and ensure a better prognosis.

BASELINE:
The outlook for Tuberculosis (TB) depends heavily on the **stage of the disease**, the **response to treatment**, and the **compliance with medication**. Here is a breakdown of what to expect:

### 1. If the Disease is Controllable (Most Common Scenario)
If you are diagnosed with **active pulmonary TB** (in the lungs) and you start treatment immediately, the prognosis is generally excellent.
*   **High Cure Rate:** With

## 14. Zapis i zachowanie wyników eksperymentu

Po zakończeniu eksperymentu zapisano adapter LoRA oraz pliki zawierające wyniki ewaluacji modelu bazowego i modelu po dostrojeniu.

Dzięki temu możliwe jest ponowne wykorzystanie zapisanych wyników bez konieczności powtarzania całego procesu treningowego.